# Hugging Face: готовые CRNN и CNN + Transformer

In [ ]:
from pathlib import Path
import sys, random
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'captcha_lab').is_dir())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
random.seed(42)
import numpy as np
import pandas as pd
np.random.seed(42)
from captcha_lab.data import image_manifest, ensure_labels, crop_image, save_json
from captcha_lab.benchmark import run_benchmark, save_run, evaluate
print('Python:', sys.version.split()[0])

Сравниваем четыре небольшие обученные модели Graf-J: base и finetuned каждой архитектуры.
Base обучены на hammer888/captcha-data, finetuned дополнительно на другом генераторе.
Ревизии зафиксированы в `research/hf_lock.json`. Используется просмотренный локальный код и safetensors,
без `trust_remote_code=True`. Скорость автора на RTX A6000 не переносится на наш CPU.

In [ ]:
from captcha_lab.hf_models import load_graf
manifest = image_manifest(); assert not manifest.empty
MODELS = ['Graf-J/captcha-crnn-base','Graf-J/captcha-crnn-finetuned',
          'Graf-J/captcha-conv-transformer-base','Graf-J/captcha-conv-transformer-finetuned']

In [ ]:
frames, failures = [], {}
for model_id in MODELS:
    print('Модель:',model_id,flush=True)
    try:
        recognizer = load_graf(model_id)
        frame = run_benchmark(manifest,recognizer,model_id,variants=('rgb','gray'),repeats=3)
        frame.attrs.update(revision=recognizer.revision,parameters=recognizer.parameters)
        save_run(frame,model_id.replace('/','--')+'.csv'); frames.append(frame)
        del recognizer
    except Exception as exc:
        failures[model_id] = type(exc).__name__
save_json(ROOT/'results/hf_failures.json',failures)
print('Ошибки загрузки/запуска:',failures)
assert frames, 'Нет успешных запусков'
predictions = pd.concat(frames,ignore_index=True)
display(evaluate(predictions,ensure_labels(manifest)))

Дополнительный тяжёлый кандидат: anuashok/ocr-captcha-v3, fine-tuned TrOCR.
Он найден при поиске, но по умолчанию отключён: скачивание весов может превышать 1 ГБ.
Не следует смешивать CER автора и точность полного совпадения на наших данных.

In [ ]:
RUN_LARGE_TROCR = False
if RUN_LARGE_TROCR:
    from captcha_lab.hf_models import load_trocr
    predictor = load_trocr()
    frame = run_benchmark(manifest,predictor,'anuashok/ocr-captcha-v3',repeats=1)
    save_run(frame,'trocr-v3.csv')
    display(evaluate(frame,ensure_labels(manifest)))
else:
    print('TrOCR не запускался: RUN_LARGE_TROCR=False')